In [1]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import (lfilter, firwin, remez, iirdesign, iirnotch, freqz, welch,
                          group_delay, tf2sos, sosfilt)
# Anggota Kelompok :
# 1. Christama Ezra Yudianto    (245150307111009)
# 2. Abdillah Fatah Suryanto    (245150300111008)
# 3. Ivang Arfandia             (245150307111003)
# 4. Jahzeel Erason Nicodemus   (245150300111002)

ROOT = Path(r"C:\Users\Erick W\OneDrive\Documents\Python\Mini Project 06")
STATUS_PATH = ROOT / "project_status.json"
integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

## 1. Validasi handoff MP5

In [2]:
with open(STATUS_PATH, "r", encoding="utf-8") as f:
    status = json.load(f)
assert status.get("status") in ("ready_for_mp06", "ready_for_mp07"), "Jalankan Mini Project 5 terlebih dahulu."
input_wav = ROOT / status["filter_input_hum"]
fs, x_pcm = wavfile.read(input_wav)
if x_pcm.ndim > 1:
    x_pcm = x_pcm[:, 0]
if np.issubdtype(x_pcm.dtype, np.integer):
    x = x_pcm.astype(np.float64) / np.iinfo(x_pcm.dtype).max
else:
    x = x_pcm.astype(np.float64)

## 2. Acceptance criteria

In [3]:
spec = {
    "sample_rate_hz": int(fs),
    "target_interference_hz": 50.0,
    "hum_attenuation_min_db": 25.0,
    "speech_band_hz": [150.0, 4000.0],
    "speech_spectral_distortion_max_db_rms": 1.5,
    "additional_delay_max_ms": 10.0,
    "multiplies_per_sample_target": 30,
    "selection_rule": "meet quality first, then minimize distortion + compute cost + delay",
}
config_dir = ROOT / "config"
feat_dir = ROOT / "features/intermediate"
out_audio = ROOT / "data/processed/mp06_design"
out_met = ROOT / "results/metrics"
out_fig = ROOT / "results/figures"
firmware = ROOT / "firmware"
for d in [config_dir, feat_dir, out_audio, out_met, out_fig, firmware]:
    d.mkdir(parents=True, exist_ok=True)
with open(config_dir / "filter_design_spec.json", "w", encoding="utf-8") as f:
    json.dump(spec, f, indent=2)

## 3. Kandidat desain formal

In [4]:
candidates = {}
# A. FIR window: baseline terkontrol
b = firwin(161, 115.0, pass_zero=False, window="hamming", fs=fs)
candidates["fir_hamming_hp"] = {"b": b, "a": np.array([1.0]), "type": "FIR", "delay_ms": (len(b)-1)/2/fs*1000}
# B. FIR Parks-McClellan: stopband 0-80 Hz, passband >=150 Hz
b = remez(161, [0, 80, 150, fs/2], [0, 1], weight=[10, 1], fs=fs, maxiter=1000)
candidates["fir_remez_hp"] = {"b": b, "a": np.array([1.0]), "type": "FIR", "delay_ms": (len(b)-1)/2/fs*1000}
# C. IIR Elliptic high-pass sesuai spesifikasi formal
b, a = iirdesign(wp=150, ws=80, gpass=1, gstop=40, fs=fs, ftype="ellip", output="ba")
candidates["iir_ellip_hp"] = {"b": b, "a": a, "type": "IIR", "delay_ms": 0.0}
# D-E-F. Notch terlokalisasi dengan tiga Q untuk kebutuhan hum 50 Hz
for q in [20.0, 30.0, 40.0]:
    b, a = iirnotch(50.0, Q=q, fs=fs)
    candidates[f"iir_notch_q{int(q)}"] = {"b": b, "a": a, "type": "IIR", "delay_ms": 0.0}

## 4. Fungsi metrik

In [5]:
f_psd, P_in = welch(x, fs=fs, nperseg=min(4096, len(x)))

def band_power(P, f, lo, hi):
    m = (f >= lo) & (f <= hi)
    return float(integrate(P[m], f[m])) if np.any(m) else 0.0

def speech_distortion_db(y):
    f, P_y = welch(y, fs=fs, nperseg=min(4096, len(y)))
    m = (f >= spec["speech_band_hz"][0]) & (f <= spec["speech_band_hz"][1])
    in_db = 10*np.log10(P_in[m] + 1e-18)
    y_db = 10*np.log10(P_y[m] + 1e-18)
    return float(np.sqrt(np.mean((y_db - in_db)**2)))

p_hum_in = band_power(P_in, f_psd, 45, 55)
metrics = {}
outputs = {}
for name, c in candidates.items():
    y = lfilter(c["b"], c["a"], x)
    outputs[name] = y
    f_y, P_y = welch(y, fs=fs, nperseg=min(4096, len(y)))
    p_hum_y = band_power(P_y, f_y, 45, 55)
    attenuation_db = 10*np.log10((p_hum_in + 1e-18)/(p_hum_y + 1e-18))
    dist_db = speech_distortion_db(y)
    # Direct-form estimate: numerator + feedback multiplies per sample
    mult = len(c["b"]) + max(0, len(c["a"])-1)
    if c["type"] == "FIR":
        delay_ms = float(c["delay_ms"])
    else:
        gd_f = np.linspace(spec["speech_band_hz"][0], spec["speech_band_hz"][1], 256)
        _, gd_samples = group_delay((c["b"], c["a"]), w=gd_f, fs=fs)
        gd_samples = gd_samples[np.isfinite(gd_samples)]
        delay_ms = float(np.percentile(np.abs(gd_samples), 95) / fs * 1000) if len(gd_samples) else 0.0
    meets = (
        attenuation_db >= spec["hum_attenuation_min_db"] and
        dist_db <= spec["speech_spectral_distortion_max_db_rms"] and
        delay_ms <= spec["additional_delay_max_ms"]
    )
    # quality dominates; compute cost is secondary
    score = dist_db + 0.01*mult + 0.05*delay_ms - 0.005*attenuation_db
    metrics[name] = {
        "hum_attenuation_db": float(attenuation_db),
        "speech_spectral_distortion_db_rms": float(dist_db),
        "estimated_multiplies_per_sample": int(mult),
        "additional_delay_ms": delay_ms,
        "meets_quality_requirements": bool(meets),
        "selection_score": float(score),
        "b_len": int(len(c["b"])),
        "a_len": int(len(c["a"])),
    }

eligible = [n for n in candidates if metrics[n]["meets_quality_requirements"]]
if eligible:
    selected = min(eligible, key=lambda n: metrics[n]["selection_score"])
else:
    # Fallback pedagogis: pilih skor terbaik dan tandai bahwa requirement belum terpenuhi
    selected = min(candidates, key=lambda n: metrics[n]["selection_score"])
final = candidates[selected]
y_final = outputs[selected]
peak = np.max(np.abs(y_final))
if peak > 0.999:
    y_final = y_final / peak * 0.98

## 5. Simpan audio final PCM16

In [6]:
wav_out = out_audio / "nyala_001_filter_final.wav"
wavfile.write(wav_out, fs, np.int16(np.clip(y_final, -1, 1) * 32767))

## 6. Simpan koefisien kandidat + final

In [7]:
npz_all = {}
for name, c in candidates.items():
    npz_all[f"{name}_b"] = np.asarray(c["b"], dtype=np.float64)
    npz_all[f"{name}_a"] = np.asarray(c["a"], dtype=np.float64)
np.savez(feat_dir / "mp06_filter_candidates.npz", **npz_all)
np.savez(feat_dir / "mp06_filter_final.npz",
         b=np.asarray(final["b"], dtype=np.float64),
         a=np.asarray(final["a"], dtype=np.float64),
         sample_rate_hz=np.array([fs]),
         selected_name=np.array([selected]))

## 7. Test vector untuk firmware (maks. 0,25 s)

In [8]:
N = min(len(x), int(0.25*fs))
test_x = x[:N]
test_y = lfilter(final["b"], final["a"], test_x)
np.savez(feat_dir / "mp06_filter_test_vector.npz",
         input=test_x.astype(np.float32),
         output=test_y.astype(np.float32),
         sample_rate_hz=np.array([fs]))

## 8. Export header C float32

In [9]:
b = np.asarray(final["b"], dtype=np.float32)
a = np.asarray(final["a"], dtype=np.float32)

def c_array(name, arr):
    vals = ", ".join(f"{v:#.9g}f" for v in arr)
    return f"static const float {name}[{len(arr)}] = {{ {vals} }};"

header = "\n".join([
    "#pragma once",
    "/* Auto-generated by Mini Project 6 */",
    f"#define AUDIO_FS_HZ {fs}",
    f"#define FILTER_B_LEN {len(b)}",
    f"#define FILTER_A_LEN {len(a)}",
    c_array("FILTER_B", b),
    c_array("FILTER_A", a),
    "/* Keep state between audio blocks; do not reset per frame. */",
])
with open(firmware / "mp06_filter_coefficients.h", "w", encoding="utf-8") as f:
    f.write(header + "\n")

## 9. Konfigurasi final + metrik

In [10]:
final_config = {
    "selected_filter": selected,
    "sample_rate_hz": int(fs),
    "b_length": int(len(final["b"])),
    "a_length": int(len(final["a"])),
    "input_audio": str(input_wav.relative_to(ROOT)),
    "output_audio": str(wav_out.relative_to(ROOT)),
    "coefficients": "features/intermediate/mp06_filter_final.npz",
    "firmware_header": "firmware/mp06_filter_coefficients.h",
    "test_vector": "features/intermediate/mp06_filter_test_vector.npz",
}
with open(config_dir / "filter_final_config.json", "w", encoding="utf-8") as f:
    json.dump(final_config, f, indent=2)
report = {"spec": spec, "selected_filter": selected, "candidates": metrics}
with open(out_met / "mp06_filter_design_metrics.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

## 10. Plot respons semua kandidat

In [11]:
fig, ax = plt.subplots(1, 1, figsize=(9, 5.5))
for name, c in candidates.items():
    f_r, H = freqz(c["b"], c["a"], worN=8192, fs=fs)
    ax.plot(f_r, 20*np.log10(np.abs(H)+1e-12), label=name, linewidth=1.0)
ax.axvline(50, linestyle="--", linewidth=0.8)
ax.set_xlim(0, 700); ax.set_ylim(-90, 5)
ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Magnitude (dB)")
ax.set_title("Mini Project 6 - candidate filter responses")
ax.grid(alpha=0.2); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(out_fig / "mp06_filter_candidates.png", dpi=180); plt.close()

## 11. Plot efek final pada PSD

In [12]:
f_y, P_y = welch(y_final, fs=fs, nperseg=min(4096, len(y_final)))
fig, ax = plt.subplots(2, 1, figsize=(9, 7))
show = min(len(x), int(0.2*fs)); t = np.arange(show)/fs
ax[0].plot(t, x[:show], label="input + hum", linewidth=0.8)
ax[0].plot(t, y_final[:show], label=f"final: {selected}", linewidth=0.8)
ax[0].set_xlabel("Time (s)"); ax[0].set_title("Waveform"); ax[0].legend(); ax[0].grid(alpha=0.2)
ax[1].plot(f_psd, 10*np.log10(P_in+1e-18), label="input")
ax[1].plot(f_y, 10*np.log10(P_y+1e-18), label="final")
ax[1].set_xlim(0, 1000); ax[1].set_xlabel("Frequency (Hz)"); ax[1].set_ylabel("PSD (dB/Hz)")
ax[1].set_title("PSD before / after final filter"); ax[1].legend(); ax[1].grid(alpha=0.2)
plt.tight_layout(); plt.savefig(out_fig / "mp06_filter_final_effect.png", dpi=180); plt.close()

## 12. Handoff ke MP7

In [13]:
status.update({
    "status": "ready_for_mp07",
    "last_completed_mini_project": 6,
    "filter_design_spec": "config/filter_design_spec.json",
    "filter_final_config": "config/filter_final_config.json",
    "filter_final_coefficients": "features/intermediate/mp06_filter_final.npz",
    "filter_test_vector": "features/intermediate/mp06_filter_test_vector.npz",
    "filter_final_audio": "data/processed/mp06_design/nyala_001_filter_final.wav",
    "filter_design_metrics": "results/metrics/mp06_filter_design_metrics.json",
    "mp07_input_audio": "data/processed/mp06_design/nyala_001_filter_final.wav",
})
with open(STATUS_PATH, "w", encoding="utf-8") as f:
    json.dump(status, f, indent=2)

print("Mini Project 6 selesai")
print("Selected filter:", selected)
for name in metrics:
    m = metrics[name]
    print(name, "atten=", round(m["hum_attenuation_db"], 2),
          "dist=", round(m["speech_spectral_distortion_db_rms"], 3),
          "mult=", m["estimated_multiplies_per_sample"],
          "pass=", m["meets_quality_requirements"])
print("Status: ready_for_mp07")

Mini Project 6 selesai
Selected filter: iir_ellip_hp
fir_hamming_hp atten= 14.54 dist= 0.385 mult= 161 pass= False
fir_remez_hp atten= 32.77 dist= 1.539 mult= 161 pass= False
iir_ellip_hp atten= 50.27 dist= 0.874 mult= 9 pass= True
iir_notch_q20 atten= 19.62 dist= 0.0 mult= 5 pass= False
iir_notch_q30 atten= 16.42 dist= 0.0 mult= 5 pass= False
iir_notch_q40 atten= 14.45 dist= 0.0 mult= 5 pass= False
Status: ready_for_mp07


### FITUR TAMBAHAN - FIRMWARE READINESS & ROBUSTNESS ANALYSIS
##### Deskripsi Fitur Tambahan (Sesuai Karakteristik Mini Project 6):
##### 1. Transien & Redaman Steady-State: settling time tiap kandidat dan redaman hum
#####    dihitung hanya pada bagian sinyal setelah t = 1,5 s (filter sudah settling).
##### 2. Verifikasi Aritmetika float32 vs float64: simulasi presisi MCU pada test vector.
##### 3. Konversi Second-Order Sections (SOS) + header CMSIS-DSP (arm_biquad_cascade_df1_f32).
##### 4. Sensitivitas Kuantisasi Koefisien: radius pole direct form vs SOS
#####    pada berbagai jumlah bit pecahan (persiapan fixed-point).
##### 5. Visualisasi Perbandingan Kandidat terhadap Acceptance Criteria (2x2).

In [14]:
print("\n" + "=" * 60)
print("FITUR TAMBAHAN - FIRMWARE READINESS & ROBUSTNESS ANALYSIS")
print("=" * 60)

# 1. TRANSIEN & REDAMAN STEADY-STATE SETIAP KANDIDAT
ss_start = int(1.5 * fs)
if len(x) - ss_start < 4096:
    ss_start = len(x) // 2
nps_ss = min(4096, len(x) - ss_start)
f_ss, P_in_ss = welch(x[ss_start:], fs=fs, nperseg=nps_ss)
p_hum_in_ss = band_power(P_in_ss, f_ss, 45, 55)
transient = {}
for name, c in candidates.items():
    _, P_y_ss = welch(outputs[name][ss_start:], fs=fs, nperseg=nps_ss)
    att_ss = 10*np.log10((p_hum_in_ss + 1e-18)/(band_power(P_y_ss, f_ss, 45, 55) + 1e-18))
    if c["type"] == "IIR":
        r_max = float(np.max(np.abs(np.roots(c["a"]))))
        t60_ms = float(1000*np.log(1e-3)/np.log(r_max)/fs) if r_max < 1 else float("inf")
    else:
        r_max = 0.0
        t60_ms = float(1000*(len(c["b"]) - 1)/fs)
    transient[name] = {
        "max_pole_radius": r_max,
        "settling_60db_ms": t60_ms,
        "steady_state_attenuation_db": float(att_ss),
        "meets_attenuation_steady_state": bool(att_ss >= spec["hum_attenuation_min_db"]),
    }

# 2. VERIFIKASI ARITMETIKA FLOAT32 VS REFERENSI FLOAT64
b64 = np.asarray(final["b"], dtype=np.float64)
a64 = np.asarray(final["a"], dtype=np.float64)
y_ref64 = lfilter(b64, a64, test_x)
y_f32 = lfilter(b64.astype(np.float32), a64.astype(np.float32), test_x.astype(np.float32))
err_f32 = y_ref64 - np.asarray(y_f32, dtype=np.float64)
f32_max_err = float(np.max(np.abs(err_f32)))
f32_snr_db = float(10*np.log10((np.mean(y_ref64**2) + 1e-20)/(np.mean(err_f32**2) + 1e-20)))

# 3. KONVERSI KE SECOND-ORDER SECTIONS (SOS) & HEADER CMSIS-DSP
final_is_iir = len(a64) > 1
sos_info = {"applicable": final_is_iir}
if final_is_iir:
    sos = tf2sos(b64, a64)
    n_stage = int(sos.shape[0])
    y_sos32 = sosfilt(sos.astype(np.float32), test_x.astype(np.float32))
    err_sos = y_ref64 - np.asarray(y_sos32, dtype=np.float64)
    sos_snr_db = float(10*np.log10((np.mean(y_ref64**2) + 1e-20)/(np.mean(err_sos**2) + 1e-20)))
    cmsis_coeffs = []
    for s in sos:
        b0, b1, b2, a0, a1, a2 = s
        cmsis_coeffs += [b0, b1, b2, -a1, -a2]
    cmsis_coeffs = np.asarray(cmsis_coeffs, dtype=np.float32)
    header_sos = "\n".join([
        "#pragma once",
        "/* Fitur tambahan Mini Project 6: koefisien SOS untuk CMSIS-DSP */",
        "/* Fungsi: arm_biquad_cascade_df1_f32, urutan per stage {b0, b1, b2, -a1, -a2} */",
        f"/* Filter: {selected} */",
        f"#define AUDIO_FS_HZ {fs}",
        f"#define FILTER_NUM_STAGES {n_stage}",
        c_array("FILTER_SOS_CMSIS", cmsis_coeffs),
        "/* State: float state[4 * FILTER_NUM_STAGES]; jangan reset antar blok audio. */",
    ])
    sos_header_path = firmware / "mp06_filter_sos_cmsis.h"
    with open(sos_header_path, "w", encoding="utf-8") as f:
        f.write(header_sos + "\n")
    sos_info.update({
        "num_stages": n_stage,
        "multiplies_per_sample": 5 * n_stage,
        "float32_snr_vs_float64_db": sos_snr_db,
        "header": "firmware/mp06_filter_sos_cmsis.h",
    })

# 4. SENSITIVITAS KUANTISASI KOEFISIEN (DIRECT FORM VS SOS)
def quantize(v, frac_bits):
    step = 2.0 ** (-frac_bits)
    return np.round(np.asarray(v, dtype=np.float64) / step) * step

bits_list = [8, 10, 12, 14, 16, 20, 24]
quant_results = []
if final_is_iir:
    for nb in bits_list:
        r_df = float(np.max(np.abs(np.roots(quantize(a64, nb)))))
        sos_q = quantize(sos, nb)
        r_sos = float(max(np.max(np.abs(np.roots(s[3:]))) for s in sos_q))
        quant_results.append({
            "fractional_bits": nb,
            "pole_radius_direct_form": r_df,
            "pole_radius_sos": r_sos,
            "stable_direct_form": bool(r_df < 1.0),
            "stable_sos": bool(r_sos < 1.0),
        })

# 5. SIMPAN METRIK FITUR TAMBAHAN KE JSON
ext_metrics = {
    "feature_extension": "Firmware Readiness & Robustness Analysis",
    "selected_filter": selected,
    "steady_state_window_start_s": float(ss_start / fs),
    "transient_and_steady_state": transient,
    "float32_verification": {
        "max_abs_error": f32_max_err,
        "snr_float32_vs_float64_db": f32_snr_db,
    },
    "sos_conversion": sos_info,
    "coefficient_quantization": quant_results,
}
ext_met_path = out_met / "mp06_extension_firmware_metrics.json"
with open(ext_met_path, "w", encoding="utf-8") as f:
    json.dump(ext_metrics, f, indent=2)

# 6. VISUALISASI FITUR TAMBAHAN (4 SUBPLOT)
names = list(candidates.keys())
short = [n.replace("iir_", "").replace("fir_", "") for n in names]
idx = np.arange(len(names))
pass_colors = ["seagreen" if metrics[n]["meets_quality_requirements"] else "gray" for n in names]
fig_ext, ax_ext = plt.subplots(2, 2, figsize=(12, 8.5))

# Subplot 1: Redaman hum seluruh sinyal vs steady-state
att_full = [metrics[n]["hum_attenuation_db"] for n in names]
att_ss = [transient[n]["steady_state_attenuation_db"] for n in names]
ax_ext[0, 0].bar(idx - 0.2, att_full, width=0.4, label="Seluruh sinyal (metrik modul)", color="steelblue")
ax_ext[0, 0].bar(idx + 0.2, att_ss, width=0.4, label=f"Steady-state (t > {ss_start/fs:.1f} s)", color="darkorange")
ax_ext[0, 0].axhline(spec["hum_attenuation_min_db"], color="crimson", linestyle="--", linewidth=1.0, label="Syarat 25 dB")
ax_ext[0, 0].set_xticks(idx); ax_ext[0, 0].set_xticklabels(short, rotation=20, fontsize=8)
ax_ext[0, 0].set_title("Extension 1: Redaman Hum 45-55 Hz (Pengaruh Transien)")
ax_ext[0, 0].set_ylabel("Redaman (dB)")
ax_ext[0, 0].legend(fontsize=7); ax_ext[0, 0].grid(axis="y", alpha=0.25)

# Subplot 2: Distorsi pita ucapan
dist_vals = [metrics[n]["speech_spectral_distortion_db_rms"] for n in names]
ax_ext[0, 1].bar(idx, dist_vals, color=pass_colors)
ax_ext[0, 1].axhline(spec["speech_spectral_distortion_max_db_rms"], color="crimson", linestyle="--", linewidth=1.0, label="Batas 1,5 dB")
ax_ext[0, 1].set_xticks(idx); ax_ext[0, 1].set_xticklabels(short, rotation=20, fontsize=8)
ax_ext[0, 1].set_title(f"Extension 2: Distorsi 150-4000 Hz | Terpilih: {selected}")
ax_ext[0, 1].set_ylabel("Distorsi (dB RMS)")
ax_ext[0, 1].legend(fontsize=7); ax_ext[0, 1].grid(axis="y", alpha=0.25)

# Subplot 3: Biaya komputasi
mult_vals = [metrics[n]["estimated_multiplies_per_sample"] for n in names]
ax_ext[1, 0].bar(idx, mult_vals, color=pass_colors)
ax_ext[1, 0].axhline(spec["multiplies_per_sample_target"], color="crimson", linestyle="--", linewidth=1.0, label="Target 30")
ax_ext[1, 0].set_yscale("log")
ax_ext[1, 0].set_xticks(idx); ax_ext[1, 0].set_xticklabels(short, rotation=20, fontsize=8)
ax_ext[1, 0].set_title("Extension 3: Perkalian per Sampel (hijau = lolos kualitas)")
ax_ext[1, 0].set_ylabel("Perkalian / sampel (log)")
ax_ext[1, 0].legend(fontsize=7); ax_ext[1, 0].grid(axis="y", alpha=0.25)

# Subplot 4: Sensitivitas kuantisasi koefisien
if final_is_iir and quant_results:
    nbs = [q["fractional_bits"] for q in quant_results]
    ax_ext[1, 1].plot(nbs, [q["pole_radius_direct_form"] for q in quant_results], "o-", label="Direct form")
    ax_ext[1, 1].plot(nbs, [q["pole_radius_sos"] for q in quant_results], "s-", label="SOS (biquad)")
    ax_ext[1, 1].axhline(1.0, color="crimson", linestyle="--", linewidth=1.0, label="Batas stabil |p| = 1")
    ax_ext[1, 1].set_title("Extension 4: Radius Pole Maksimum vs Bit Pecahan Koefisien")
    ax_ext[1, 1].set_xlabel("Jumlah bit pecahan")
    ax_ext[1, 1].set_ylabel("Radius pole maksimum")
    ax_ext[1, 1].legend(fontsize=7)
else:
    ax_ext[1, 1].text(0.5, 0.5, "Filter final FIR: tanpa pole,\nselalu stabil terhadap kuantisasi",
                      ha="center", va="center", fontsize=10)
    ax_ext[1, 1].set_title("Extension 4: Sensitivitas Kuantisasi")
ax_ext[1, 1].grid(alpha=0.25)

fig_ext.tight_layout()
ext_fig_path = out_fig / "mp06_extension_firmware_analysis.png"
fig_ext.savefig(ext_fig_path, dpi=180)
plt.close(fig_ext)

# 7. OUTPUT TERMINAL FITUR TAMBAHAN
print(f"{'Kandidat':<16}{'Settling(ms)':>13}{'Att penuh':>11}{'Att steady':>12}{'Lolos SS':>10}")
for n in names:
    tr = transient[n]
    print(f"{n:<16}{tr['settling_60db_ms']:>13.1f}{metrics[n]['hum_attenuation_db']:>11.2f}"
          f"{tr['steady_state_attenuation_db']:>12.2f}{str(tr['meets_attenuation_steady_state']):>10}")
print("Float32 vs float64 SNR       :", round(f32_snr_db, 2), "dB | max error:", f"{f32_max_err:.3e}")
if final_is_iir:
    print("Jumlah stage SOS             :", sos_info["num_stages"])
    print("SOS float32 SNR              :", round(sos_info["float32_snr_vs_float64_db"], 2), "dB")
    for q in quant_results:
        print(f"Kuantisasi {q['fractional_bits']:>2} bit: |p| DF = {q['pole_radius_direct_form']:.6f} "
              f"(stabil {q['stable_direct_form']}), |p| SOS = {q['pole_radius_sos']:.6f} (stabil {q['stable_sos']})")
print("Extension Metrics File       :", ext_met_path)
print("Extension Figure File        :", ext_fig_path)
print("=" * 60)


FITUR TAMBAHAN - FIRMWARE READINESS & ROBUSTNESS ANALYSIS
Kandidat         Settling(ms)  Att penuh  Att steady  Lolos SS
fir_hamming_hp           10.0      14.54       14.54     False
fir_remez_hp             10.0      32.77       32.76      True
iir_ellip_hp             69.6      50.27       50.26      True
iir_notch_q20           879.5      19.62       41.77      True
iir_notch_q30          1319.3      16.42       40.03      True
iir_notch_q40          1759.0      14.45       38.75      True
Float32 vs float64 SNR       : 10.65 dB | max error: 3.459e-04
Jumlah stage SOS             : 2
SOS float32 SNR              : 73.12 dB
Kuantisasi  8 bit: |p| DF = 1.151712 (stabil False), |p| SOS = 0.994123 (stabil True)
Kuantisasi 10 bit: |p| DF = 1.000000 (stabil False), |p| SOS = 0.993632 (stabil True)
Kuantisasi 12 bit: |p| DF = 1.000000 (stabil False), |p| SOS = 0.993755 (stabil True)
Kuantisasi 14 bit: |p| DF = 1.000000 (stabil True), |p| SOS = 0.993816 (stabil True)
Kuantisasi 16 bit: |p